# Telco Customer Churn Prediction
## Portfolio Edition — Machine Learning, Model Evaluation and Retention Decision Support

**Author:** Sebetja Success Phiri

This notebook develops an end-to-end customer churn workflow using the IBM Telco Customer Churn dataset.

The project combines:

- data quality assessment and preprocessing;
- exploratory data analysis;
- Logistic Regression and Random Forest classification;
- accuracy, precision, recall, F1, ROC-AUC and PR-AUC;
- five-fold stratified cross-validation;
- confusion-matrix analysis;
- feature importance;
- validation-based decision-threshold optimisation; and
- an illustrative business-cost framework for retention decisions.

The objective is not simply to maximise predictive accuracy. In churn management, a false negative represents a customer who actually leaves but was not identified as at risk. Model evaluation therefore focuses on the trade-off between **recall, precision, false negatives and intervention cost**.


## 1. Imports and Project Configuration

The notebook uses only widely adopted Python data-science libraries so that the analysis remains easy to reproduce.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

RANDOM_STATE = 42
SAVE_FIGURES = True

# Support running the notebook either from the repository root
# or from the notebooks/ directory.
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd.parent

DATA_PATH = PROJECT_ROOT / "data" / "Telco-Customer-Churn.csv"
FIGURES_DIR = PROJECT_ROOT / "images"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Expected data file: {DATA_PATH.resolve()}")


## 2. Load the Dataset

The IBM Telco Customer Churn dataset contains one row per customer and combines demographic, service, contract and billing attributes with the churn outcome.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Download the IBM Telco Customer Churn CSV and save it as "
        f"'{DATA_PATH}'. See data/README.md for the source."
    )

telecom_df = pd.read_csv(DATA_PATH)

print("DATASET LOADED")
print("-" * 60)
print(f"Rows:    {telecom_df.shape[0]:,}")
print(f"Columns: {telecom_df.shape[1]}")
telecom_df.head()


## 3. Data Quality Assessment

Before modelling, inspect the dataset structure, data types, missing values and target distribution. `TotalCharges` is stored as text in the source data and contains blank values that become missing values when converted to numeric form.


In [ ]:
print("DATA TYPES")
display(telecom_df.dtypes.to_frame("dtype"))

print("\nRAW MISSING VALUES")
display(telecom_df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nTARGET DISTRIBUTION")
display(
    telecom_df["Churn"]
    .value_counts(dropna=False)
    .rename_axis("Churn")
    .to_frame("Count")
)


## 4. Data Cleaning and Target Preparation

Cleaning follows the original bootcamp workflow:

1. Convert `TotalCharges` to numeric.
2. Remove rows where conversion reveals missing values.
3. Remove `customerID`, which is an identifier rather than a predictive feature.
4. Encode `Churn` as `1` for Yes and `0` for No.


In [ ]:
telecom_clean = telecom_df.copy()

telecom_clean["TotalCharges"] = pd.to_numeric(
    telecom_clean["TotalCharges"],
    errors="coerce",
)

missing_total_charges = telecom_clean["TotalCharges"].isna().sum()
telecom_clean = telecom_clean.dropna(subset=["TotalCharges"]).copy()

telecom_clean = telecom_clean.drop(columns=["customerID"])
telecom_clean["Churn"] = telecom_clean["Churn"].map({"No": 0, "Yes": 1}).astype(int)

print("CLEANING SUMMARY")
print("-" * 60)
print(f"Missing TotalCharges identified: {missing_total_charges}")
print(f"Rows after cleaning:             {len(telecom_clean):,}")
print(f"Columns after ID removal:        {telecom_clean.shape[1]}")
print(f"Remaining missing values:        {telecom_clean.isna().sum().sum()}")


## 5. Exploratory Data Analysis

The following analysis examines churn prevalence and key customer characteristics before model development.


In [ ]:
churn_distribution = (
    telecom_clean["Churn"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("CHURN DISTRIBUTION (%)")
display(churn_distribution.rename(index={0: "Stayed", 1: "Churned"}).to_frame("Percent"))

fig, ax = plt.subplots(figsize=(8, 5))
churn_distribution.rename(index={0: "Stayed", 1: "Churned"}).plot(kind="bar", ax=ax)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Customer Status")
ax.set_ylabel("Percent of Customers")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(FIGURES_DIR / "01_churn_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
tenure_summary = (
    telecom_clean.groupby("Churn")["tenure"]
    .agg(["count", "mean", "median", "std"])
    .rename(index={0: "Stayed", 1: "Churned"})
)

print("TENURE BY CHURN STATUS")
display(tenure_summary)

fig, ax = plt.subplots(figsize=(8, 5))
groups = [
    telecom_clean.loc[telecom_clean["Churn"] == 0, "tenure"],
    telecom_clean.loc[telecom_clean["Churn"] == 1, "tenure"],
]
ax.boxplot(groups, tick_labels=["Stayed", "Churned"], showmeans=True)
ax.set_title("Customer Tenure by Churn Status")
ax.set_xlabel("Customer Status")
ax.set_ylabel("Tenure (Months)")
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(FIGURES_DIR / "02_tenure_by_churn.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
segment_columns = ["Contract", "InternetService", "PaymentMethod"]

for column in segment_columns:
    table = (
        telecom_clean.groupby(column, observed=False)["Churn"]
        .agg(["count", "mean"])
        .rename(columns={"count": "Customers", "mean": "Churn_Rate"})
        .sort_values("Churn_Rate", ascending=False)
    )
    table["Churn_Rate"] = table["Churn_Rate"] * 100

    print(f"\nCHURN RATE BY {column.upper()}")
    display(table)


## 6. Feature Encoding and Redundancy Treatment

For direct comparison with the original bootcamp models, categorical variables are dummy encoded with `drop_first=True`.

Several generated service-state columns provide duplicate or structurally redundant information. The same seven redundant columns removed in the original analysis are removed here before modelling.


In [ ]:
telecom_encoded = pd.get_dummies(
    telecom_clean,
    drop_first=True,
    dtype=int,
)

redundant_features = [
    "OnlineSecurity_No internet service",
    "OnlineBackup_No internet service",
    "DeviceProtection_No internet service",
    "TechSupport_No internet service",
    "StreamingTV_No internet service",
    "StreamingMovies_No internet service",
    "MultipleLines_No phone service",
]

features_to_drop = [
    feature for feature in redundant_features
    if feature in telecom_encoded.columns
]

telecom_model_data = telecom_encoded.drop(columns=features_to_drop).copy()

X = telecom_model_data.drop(columns=["Churn"])
y = telecom_model_data["Churn"]

print("MODELLING DATASET")
print("-" * 60)
print(f"Observations: {X.shape[0]:,}")
print(f"Predictors:   {X.shape[1]}")
print(f"Target:       Churn")
print(f"Removed redundant predictors: {len(features_to_drop)}")


In [ ]:
correlations = (
    telecom_model_data
    .corr(numeric_only=True)["Churn"]
    .drop("Churn")
    .sort_values()
)

print("STRONGEST ABSOLUTE CORRELATIONS WITH CHURN")
display(
    correlations
    .reindex(correlations.abs().sort_values(ascending=False).index)
    .head(15)
    .to_frame("Correlation")
)

fig, ax = plt.subplots(figsize=(10, 8))
correlations.plot(kind="barh", ax=ax)
ax.axvline(0, linewidth=1)
ax.set_title("Feature Correlations with Customer Churn")
ax.set_xlabel("Pearson Correlation with Churn")
ax.set_ylabel("Feature")
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(FIGURES_DIR / "03_feature_correlations.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Train/Test Split

A stratified 75/25 split preserves the churn-class distribution in both partitions. The held-out test set is reserved for final comparison.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    "Observations": [len(X_train), len(X_test)],
    "Share": [len(X_train) / len(X), len(X_test) / len(X)],
    "Churn_Rate": [y_train.mean(), y_test.mean()],
}, index=["Training", "Test"])

display(split_summary)

majority_baseline_accuracy = max(y_test.mean(), 1 - y_test.mean())

print(
    f"Naive majority-class accuracy baseline: "
    f"{majority_baseline_accuracy:.4f} "
    f"({majority_baseline_accuracy * 100:.2f}%)"
)


## 8. Baseline Models

### Logistic Regression

Min-Max scaling is fitted on the training data only. Logistic Regression then estimates the probability of churn.

### Random Forest

The Random Forest configuration matches the original project: 2,000 trees, square-root feature sampling, a maximum of 50 leaf nodes, bootstrap sampling and out-of-bag estimation.


In [ ]:
min_max_scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    min_max_scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)

X_test_scaled = pd.DataFrame(
    min_max_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)
logistic_model.fit(X_train_scaled, y_train)

logistic_predictions = logistic_model.predict(X_test_scaled)
logistic_probabilities = logistic_model.predict_proba(X_test_scaled)[:, 1]

random_forest_model = RandomForestClassifier(
    n_estimators=2000,
    oob_score=True,
    max_features="sqrt",
    max_leaf_nodes=50,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_forest_model.fit(X_train, y_train)

random_forest_predictions = random_forest_model.predict(X_test)
random_forest_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

print("Models trained successfully.")
print(f"Random Forest OOB score: {random_forest_model.oob_score_:.4f}")


## 9. Comprehensive Model Evaluation

Accuracy is reported for context, but it is not sufficient for this imbalanced classification problem. The portfolio evaluation therefore adds:

- **Precision** — how often a churn prediction is correct;
- **Recall** — how many actual churners are identified;
- **F1** — harmonic balance between precision and recall;
- **ROC-AUC** — ranking performance across classification thresholds;
- **PR-AUC / Average Precision** — precision-recall performance, particularly useful with class imbalance; and
- the four confusion-matrix counts.


In [ ]:
def evaluate_classifier(name, y_true, predictions, probabilities):
    cm = confusion_matrix(y_true, predictions)
    tn, fp, fn, tp = cm.ravel()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "PR_AUC": average_precision_score(y_true, probabilities),
        "True_Negative": tn,
        "False_Positive": fp,
        "False_Negative": fn,
        "True_Positive": tp,
    }

model_results = pd.DataFrame([
    evaluate_classifier(
        "Logistic Regression",
        y_test,
        logistic_predictions,
        logistic_probabilities,
    ),
    evaluate_classifier(
        "Random Forest",
        y_test,
        random_forest_predictions,
        random_forest_probabilities,
    ),
])

metric_columns = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
display(
    model_results.style.format({
        column: "{:.3f}" for column in metric_columns
    })
)


In [ ]:
for name, predictions in [
    ("Logistic Regression", logistic_predictions),
    ("Random Forest", random_forest_predictions),
]:
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        display_labels=["Stayed", "Churned"],
        values_format="d",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{name} — Confusion Matrix")
    plt.tight_layout()

    if SAVE_FIGURES:
        filename = name.lower().replace(" ", "_")
        fig.savefig(
            FIGURES_DIR / f"04_{filename}_confusion_matrix.png",
            dpi=160,
            bbox_inches="tight",
        )
    plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, probabilities in [
    ("Logistic Regression", logistic_probabilities),
    ("Random Forest", random_forest_probabilities),
]:
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc_value = roc_auc_score(y_test, probabilities)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", label="No-skill reference")
ax.set_title("ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(FIGURES_DIR / "05_roc_curves.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, probabilities in [
    ("Logistic Regression", logistic_probabilities),
    ("Random Forest", random_forest_probabilities),
]:
    precision, recall, _ = precision_recall_curve(y_test, probabilities)
    ap = average_precision_score(y_test, probabilities)
    ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

ax.set_title("Precision-Recall Curves")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIGURES_DIR / "06_precision_recall_curves.png",
        dpi=160,
        bbox_inches="tight",
    )
plt.show()


## 10. Five-Fold Cross-Validation

A single train/test split can be sensitive to the particular observations selected. Five-fold stratified cross-validation provides a more stable estimate of model performance.

For Logistic Regression, scaling is placed inside a scikit-learn pipeline so the scaler is fitted separately within each training fold. This prevents preprocessing leakage during validation.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

logistic_cv_pipeline = Pipeline([
    ("scaler", MinMaxScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

random_forest_cv_model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    max_leaf_nodes=50,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv_outputs = {}

for name, model in [
    ("Logistic Regression", logistic_cv_pipeline),
    ("Random Forest", random_forest_cv_model),
]:
    cv_result = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    cv_outputs[name] = cv_result

cv_summary_rows = []

for name, result in cv_outputs.items():
    row = {"Model": name}
    for metric in scoring:
        values = result[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()
    cv_summary_rows.append(row)

cv_summary = pd.DataFrame(cv_summary_rows)
display(cv_summary)


## 11. Random Forest Feature Importance

Feature importance is useful for understanding which predictors the Random Forest relied on most strongly. It is not evidence that a feature causes churn.


In [ ]:
feature_importance_df = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": random_forest_model.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

top_features = feature_importance_df.head(15)
display(top_features)

fig, ax = plt.subplots(figsize=(10, 7))
plot_data = top_features.sort_values("Importance")
ax.barh(plot_data["Feature"], plot_data["Importance"])
ax.set_title("Top 15 Random Forest Feature Importances")
ax.set_xlabel("Feature Importance")
ax.set_ylabel("Feature")
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIGURES_DIR / "07_random_forest_feature_importance.png",
        dpi=160,
        bbox_inches="tight",
    )
plt.show()


## 12. Validation-Based Threshold Optimisation

The default classification threshold is 0.50. That threshold is not automatically optimal for a retention problem.

To avoid choosing a threshold directly on the test set, the training partition is split again into **model-fit** and **validation** subsets. A separate Logistic Regression model is fitted on the model-fit subset, and candidate thresholds are evaluated on the validation subset.

Two threshold criteria are illustrated:

1. **Maximum F1** — balances precision and recall.
2. **Illustrative business cost** — assumes that missing an actual churner is five times as costly as contacting a customer who would have stayed.

The 5:1 cost ratio is an analytical scenario, **not a claimed real business cost**. In an organisation, this ratio should be replaced with validated financial estimates.


In [ ]:
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

threshold_scaler = MinMaxScaler()
X_fit_scaled = threshold_scaler.fit_transform(X_fit)
X_validation_scaled = threshold_scaler.transform(X_validation)

threshold_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)
threshold_model.fit(X_fit_scaled, y_fit)

validation_probabilities = threshold_model.predict_proba(X_validation_scaled)[:, 1]

thresholds = np.arange(0.10, 0.81, 0.01)
FALSE_NEGATIVE_COST = 5
FALSE_POSITIVE_COST = 1

threshold_rows = []

for threshold in thresholds:
    predictions = (validation_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_validation, predictions).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_validation, predictions, zero_division=0),
        "Recall": recall_score(y_validation, predictions, zero_division=0),
        "F1": f1_score(y_validation, predictions, zero_division=0),
        "False_Positive": fp,
        "False_Negative": fn,
        "Illustrative_Cost": (
            fp * FALSE_POSITIVE_COST
            + fn * FALSE_NEGATIVE_COST
        ),
    })

threshold_results = pd.DataFrame(threshold_rows)

best_f1_row = threshold_results.loc[threshold_results["F1"].idxmax()]
best_cost_row = threshold_results.loc[
    threshold_results["Illustrative_Cost"].idxmin()
]

print("BEST VALIDATION THRESHOLD BY F1")
display(best_f1_row.to_frame("Value"))

print("\nBEST VALIDATION THRESHOLD BY ILLUSTRATIVE COST")
display(best_cost_row.to_frame("Value"))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(threshold_results["Threshold"], threshold_results["Precision"], label="Precision")
ax.plot(threshold_results["Threshold"], threshold_results["Recall"], label="Recall")
ax.plot(threshold_results["Threshold"], threshold_results["F1"], label="F1")
ax.axvline(0.50, linestyle="--", label="Default threshold 0.50")
ax.set_title("Validation Performance Across Classification Thresholds")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIGURES_DIR / "08_threshold_tradeoffs.png",
        dpi=160,
        bbox_inches="tight",
    )
plt.show()


## 13. Test the Selected Threshold

After selecting a threshold using validation data, the Logistic Regression model is refitted on the full original training set. The selected threshold is then applied once to the held-out test set.

This maintains separation between threshold selection and final test evaluation.


In [ ]:
selected_threshold = float(best_cost_row["Threshold"])

final_scaler = MinMaxScaler()
X_train_final_scaled = final_scaler.fit_transform(X_train)
X_test_final_scaled = final_scaler.transform(X_test)

final_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)
final_logistic_model.fit(X_train_final_scaled, y_train)

final_test_probabilities = final_logistic_model.predict_proba(X_test_final_scaled)[:, 1]
threshold_predictions = (
    final_test_probabilities >= selected_threshold
).astype(int)

threshold_test_result = evaluate_classifier(
    f"Logistic Regression @ {selected_threshold:.2f}",
    y_test,
    threshold_predictions,
    final_test_probabilities,
)

comparison = pd.DataFrame([
    evaluate_classifier(
        "Logistic Regression @ 0.50",
        y_test,
        logistic_predictions,
        logistic_probabilities,
    ),
    threshold_test_result,
])

display(
    comparison.style.format({
        column: "{:.3f}"
        for column in ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
    })
)


## 14. Business Interpretation

The analysis should be interpreted as decision support rather than proof of causation.

### Early Customer Lifecycle

Customer tenure is consistently one of the strongest churn dimensions. Customers who leave tend to have substantially shorter relationships with the provider. This supports deeper investigation of onboarding, first-year service experience and early customer engagement.

### Contract Structure

Longer-term contracts are associated with lower churn than month-to-month arrangements. A company could test whether well-designed commitment incentives improve retention without damaging customer experience.

### Service Experience

Fibre-optic service appears among important churn predictors. This should trigger further investigation into pricing, reliability, support experience and customer expectations rather than a causal claim that fibre service itself creates churn.

### Billing and Payment

Monthly charges and electronic-check payment behaviour also emerge as relevant dimensions. These relationships may reflect perceived value, payment friction or customer-segment differences.

### Decision Thresholds

A retention programme should not automatically use a 0.50 probability threshold. The threshold should reflect the economics of intervention:

- What does a retention contact cost?
- What is the expected value of retaining a customer?
- What is the cost of missing a genuine churner?
- How much intervention capacity does the retention team have?

The most useful model is therefore not necessarily the model with the highest accuracy. It is the model that creates the best decision outcome under the organisation's actual constraints and costs.


## 15. Limitations and Next Steps

This portfolio project can be extended further through:

- formal hyperparameter optimisation;
- calibration analysis for churn probabilities;
- SHAP or other local/global explainability methods;
- cost estimates derived from real customer lifetime value;
- temporal validation if historical time periods are available;
- model persistence and API deployment;
- automated data validation;
- monitoring for data drift and performance drift; and
- controlled experiments to test whether retention interventions actually reduce churn.

A predictive model identifies statistical risk patterns. It does not by itself prove why a customer churns or whether a specific intervention will prevent churn.


## 16. Reference Results from the Original Bootcamp Run

Before the portfolio upgrade, the original notebook produced the following held-out test results using the same 75/25 stratified split and model configurations:

| Model | Accuracy | Precision | Recall | False Positives | False Negatives |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 80.55% | 65.28% | 57.17% | 142 | 200 |
| Random Forest | 79.52% | 66.56% | 46.04% | 108 | 252 |

The original Random Forest OOB score was **79.94%**.

These values are retained here as an audit reference. Re-running this upgraded notebook will calculate the expanded metric set, including F1, ROC-AUC, PR-AUC and cross-validation results.


---

### Portfolio Positioning

This project demonstrates the ability to move from a commercial problem to an analytical workflow, evaluate classification trade-offs, and translate model behaviour into retention decision support.

**Sebetja Success Phiri**  
Data Science & Analytics | Python | SQL | Machine Learning | Business Innovation
